In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import logging
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

from src.helper import *
from src.embedding import *
from src.model import *

In [2]:
#set config
EMBEDDED_DATA_PATH = "output/embeddings/wav2vec2/"
OUTPUT_PATH = "output/"
PREDS_PATH = "output/preds/"
MODELS_PATH = "models/"

In [3]:
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)
if not os.path.exists(PREDS_PATH):
    os.mkdir(PREDS_PATH)
if not os.path.exists(MODELS_PATH):
    os.mkdir(MODELS_PATH)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# --- Setup logging ---
logging.basicConfig(
    filename=os.path.join(OUTPUT_PATH, f"nb_04_dl_models_log.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("Main")
logger.info(f"starting DL modeling...")

### Preparing the data loader

In [6]:
logger.info(f"preparing the dataloader...")

train_loader, test_loader, holdout_loader = load_datasets(EMBEDDED_DATA_PATH, batch_size=64)

# Inspect one batch
for X, y in train_loader:
    logger.info(f"batch embeddings shape: {X.shape}")  # (batch_size, embedding_dim)
    logger.info(f"batch labels shape: {y.shape}")
    break

In [ ]:
# import os
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader, random_split
# from sklearn.metrics import classification_report

# # -----------------------
# # Custom Dataset
# # -----------------------
# class EmbeddingDataset(Dataset):
#     def __init__(self, embeddings_path, labels_path):
#         self.X = np.load(embeddings_path)   # shape: (n_samples, embedding_dim)
#         self.y = np.load(labels_path)       # shape: (n_samples,)
        
#         self.X = torch.tensor(self.X, dtype=torch.float32)
#         self.y = torch.tensor(self.y, dtype=torch.long)

#     def __len__(self):
#         return len(self.y)

#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]


# # -----------------------
# # Neural Network Model
# # -----------------------
# class AudioClassifier(nn.Module):
#     def __init__(self, input_dim, num_classes, hidden_dim=256, dropout=0.3):
#         super(AudioClassifier, self).__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim),
#             nn.BatchNorm1d(hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),

#             nn.Linear(hidden_dim, hidden_dim//2),
#             nn.BatchNorm1d(hidden_dim//2),
#             nn.ReLU(),
#             nn.Dropout(dropout),

#             nn.Linear(hidden_dim//2, num_classes)
#         )

#     def forward(self, x):
#         return self.net(x)


# # -----------------------
# # Training Function
# # -----------------------
# def train_model(model, train_loader, val_loader, device, epochs=30, lr=1e-3):
#     criterion = nn.CrossEntropyLoss()
#     optimizer = optim.Adam(model.parameters(), lr=lr)

#     best_val_loss = float("inf")
#     patience, patience_counter = 5, 0

#     for epoch in range(epochs):
#         # Training
#         model.train()
#         total_loss = 0
#         for X_batch, y_batch in train_loader:
#             X_batch, y_batch = X_batch.to(device), y_batch.to(device)

#             optimizer.zero_grad()
#             outputs = model(X_batch)
#             loss = criterion(outputs, y_batch)
#             loss.backward()
#             optimizer.step()
#             total_loss += loss.item()

#         # Validation
#         model.eval()
#         val_loss, correct, total = 0, 0, 0
#         with torch.no_grad():
#             for X_batch, y_batch in val_loader:
#                 X_batch, y_batch = X_batch.to(device), y_batch.to(device)
#                 outputs = model(X_batch)
#                 loss = criterion(outputs, y_batch)
#                 val_loss += loss.item()

#                 preds = outputs.argmax(dim=1)
#                 correct += (preds == y_batch).sum().item()
#                 total += y_batch.size(0)

#         avg_train_loss = total_loss / len(train_loader)
#         avg_val_loss = val_loss / len(val_loader)
#         val_acc = correct / total

#         print(f"Epoch [{epoch+1}/{epochs}] "
#               f"Train Loss: {avg_train_loss:.4f} "
#               f"Val Loss: {avg_val_loss:.4f} "
#               f"Val Acc: {val_acc:.4f}")

#         # Early stopping
#         if avg_val_loss < best_val_loss:
#             best_val_loss = avg_val_loss
#             torch.save(model.state_dict(), "best_model.pth")
#             patience_counter = 0
#         else:
#             patience_counter += 1
#             if patience_counter >= patience:
#                 print("Early stopping triggered.")
#                 break


# # -----------------------
# # Run Training
# # -----------------------
# if __name__ == "__main__":
#     embeddings_path = "embeddings.npy"   # replace with your file
#     labels_path = "labels.npy"           # replace with your file

#     dataset = EmbeddingDataset(embeddings_path, labels_path)

#     # 80-20 train-val split
#     val_size = int(0.2 * len(dataset))
#     train_size = len(dataset) - val_size
#     train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

#     train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
#     val_loader = DataLoader(val_dataset, batch_size=64)

#     input_dim = dataset.X.shape[1]
#     num_classes = len(torch.unique(dataset.y))

#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model = AudioClassifier(input_dim, num_classes).to(device)

#     train_model(model, train_loader, val_loader, device, epochs=50, lr=1e-3)

#     # -----------------------
#     # Evaluation
#     # -----------------------
#     model.load_state_dict(torch.load("best_model.pth"))
#     model.eval()
#     y_true, y_pred = [], []
#     with torch.no_grad():
#         for X_batch, y_batch in val_loader:
#             X_batch = X_batch.to(device)
#             outputs = model(X_batch)
#             preds = outputs.argmax(dim=1).cpu().numpy()
#             y_true.extend(y_batch.numpy())
#             y_pred.extend(preds)

#     print("\nClassification Report:\n", classification_report(y_true, y_pred))